In [1]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_DOWN
import math

In [2]:
def formatear(x, cifras):
    """Notación científica si |x| < 1 (como en papel), decimal normal si no."""
    if x is None:
        return "-"
    if x != 0 and abs(x) < 0.01:
        return a_cientifica(x, cifras)
    return f"{x:.{cifras}f}"


def imprimir_tabla(filas, cifras):
    encabezado = (f"{'n':<5}{'x_n':^25}{'f(x_n)':^25}"
                  f"{'f\'(x_n)':^25}{'Error':^25}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, x, fx, dfx, error in filas:
        print(f"{n:<5}"
              f"{formatear(x, cifras):^25}"
              f"{formatear(fx, cifras):^25}"
              f"{formatear(dfx, cifras):^25}"
              f"{formatear(error, cifras):^25}")

def imprimir_tabla1(filas, cifras):
    encabezado = (f"{'n':<5}{'x_n':^25}{'f(x_n)':^25}"
                  f"{'Error':^25}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, x, fx, error in filas:
        print(f"{n:<5}"
              f"{formatear(x, cifras):^25}"
              f"{formatear(fx, cifras):^25}"
              f"{formatear(error, cifras):^25}")
        
def imprimir_tabla2(filas, cifras):
    encabezado = (f"{'n':<5}{'p_n':^18}{'f(p_n)':^18}{'sgm(f(p_n))':^18}"
                  f"{'Error':^18}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, p_n, fp_n, sgm, error in filas:
        print(f"{n:<5}"
              f"{formatear(p_n, cifras):^18}"
              f"{formatear(fp_n, cifras):^18}"
              f"{formatear(sgm, cifras):^18}"
              f"{formatear(error, cifras):^18}")

def imprimir_tabla3(filas, cifras):
    encabezado = (f"{'n':<5}{'a':^18}{'b':^18}{'p':^18}{'sgm(f(a))':^18}"
                  f"{'sgm(f(b))':^18}{'sgm(f(p))':^18}{'Error':^18}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, a, b, p, sgma, sgmB, sgmP, error in filas:
        print(f"{n:<5}"
              f"{formatear(a, cifras):^18}"
              f"{formatear(b, cifras):^18}"
              f"{formatear(p, cifras):^18}"
              f"{formatear(sgma, cifras):^18}"
              f"{formatear(sgmB, cifras):^18}"
              f"{formatear(sgmP, cifras):^18}"
              f"{formatear(error, cifras):^18}")

In [3]:
def a_cientifica(x, cifras):
    d = Decimal(str(x))
    if d == 0:
        return f"{0:.{cifras}f}e+00"
    paso = Decimal(1).scaleb(-cifras)
    #print(paso)
    e = d.adjusted()
    #print(e)
    m = d.scaleb(-e).quantize(paso, rounding=ROUND_HALF_UP)
    if abs(m) >= 10:
        m = m / 10
        e += 1
    return f"{m}e{e:+03d}"

#a_cientifica(34.5667, 4)

In [4]:
def redondear(x, cifras):
    """
    Emula la calculadora:
      - Si |x| >= 1: redondea a `cifras` decimales (half-up).
      - Si |x| < 1: pasa a notación científica y redondea la
        mantisa a `cifras` decimales (half-up).
    Devuelve un float.
    """
    d = Decimal(str(x))
    paso = Decimal(1).scaleb(-cifras)   # 0.000001 si cifras = 6

    if d == 0 or abs(d) >= 0.01:
        return float(d.quantize(paso, rounding=ROUND_HALF_UP))

    e = d.adjusted()                   # exponente científico
    m = (d.scaleb(-e)).quantize(paso, rounding=ROUND_HALF_UP)
    # pongo -e para que la mantisa sea facil de redondear, verlo como 1.2345

    # Caso borde: 9.9999996 -> 10.000000
    if abs(m) >= 10:
        m = m / 10
        e += 1

    return float(m.scaleb(e))
    #aqui con e, la traigo a su flotante normal 0.1234567, con 7 cifras porque una de ellas es 
    #el entero


In [5]:
def truncar(x, cifras):
    """
    Emula la calculadora truncando (sin aproximar):
      - Si |x| >= 0.01: trunca a `cifras` decimales.
      - Si |x| < 0.01: pasa a notación científica y trunca la
        mantisa a `cifras` decimales.
    Devuelve un float.
    """
    d = Decimal(str(x))
    paso = Decimal(1).scaleb(-cifras)   # 0.000001 si cifras = 6

    if d == 0 or abs(d) >= 0.01:
        return float(d.quantize(paso, rounding=ROUND_DOWN))

    e = d.adjusted()                   # exponente científico
    m = (d.scaleb(-e)).quantize(paso, rounding=ROUND_DOWN)
    # pongo -e para que la mantisa sea facil de redondear, verlo como 1.2345

    # Caso borde: 9.9999996 -> 10.000000
    if abs(m) >= 10:
        m = m / 10
        e += 1

    return float(m.scaleb(e))
    #aqui con e, la traigo a su flotante normal 0.1234567, con 7 cifras porque una de ellas es 
    #el entero


In [6]:
def metodo_newton(f, df, p0, TOL, n_max, cifras=6):
    """
    Calcula una aproximación a la raíz de una función mediante el método de Newton-Raphson,
    aplicando redondeo en cada iteración para controlar la precisión de los cálculos.

    parameters:
        f (function): Función cuya raíz se desea encontrar.
        df (function): Derivada de la función f.
        p0 (float): Aproximación inicial a la raíz.
        TOL (float): Tolerancia máxima permitida para el error entre aproximaciones sucesivas.
        n_max (int): Número máximo de iteraciones permitidas.
        cifras (int): Cantidad de decimales utilizados en el redondeo. Por defecto, es 6.

    returns:
        float: Aproximación a la raíz si el procedimiento es exitoso.
        int: -1 si la derivada es cero o si no se alcanza la tolerancia dentro del número
             máximo de iteraciones.
    """
    i = 1

    p0 = redondear(p0, cifras)
    fx = redondear(f(p0), cifras)
    dfx = redondear(df(p0), cifras)

    filas = [(0, p0, fx, dfx, None)]

    while i <= n_max:
        if dfx == 0:
            imprimir_tabla(filas, cifras)
            print("Derivada igual a 0, causa indeterminación")
            return -1

        p = redondear(p0 - fx / dfx, cifras)
        error = redondear(abs(p - p0), cifras)

        # f(p) y f'(p) ya quedan listos para la siguiente fila/iteración
        fx = redondear(f(p), cifras)
        dfx = redondear(df(p), cifras)

        filas.append((i, p, fx, dfx, error))

        if error < TOL:
            imprimir_tabla(filas, cifras)
            print("\nProcedimiento exitoso!")
            return p

        i = i + 1
        p0 = p

    imprimir_tabla(filas, cifras)
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1
    

In [7]:
def secante(f, p0, p1, TOL, n_max, cifras):
    i = 2
    
    p0 = redondear(p0, cifras)
    p1 = redondear(p1, cifras)
    fp0 = redondear(f(p0), cifras)
    fp1 = redondear(f(p1), cifras)
    
    filas = [(0, p0, fp0, None)]
    filas.append((1, p1, fp1, None))
    
    while i <= n_max:
        
        denominador = (fp1 - fp0)
        
        if denominador == 0:
            imprimir_tabla1(filas, cifras)
            print("La resta entre fp1 y fp0 genera indeterminación")
            return -1
        
        p = redondear(p1 - fp1*(p1 - p0)/denominador, cifras)
        error = redondear(abs(p - p1), cifras)
        fp = redondear(f(p), cifras) #lo calculo aqui para poder mostrarlo
        
        filas.append((i, p, fp, error))
        
        if error < TOL:
            imprimir_tabla1(filas, cifras)
            print("Procedimiento Exitoso")
            return p
        
        
        p0  = p1
        fp0 = fp1
        p1  = p
        fp1 = fp
        
        i = i +1
    
    imprimir_tabla1(filas, cifras)
    
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1

In [8]:
def signum(a):
    if a < 0:
        return -1
    elif a > 0:
        return 1
    else:
        return 0

In [9]:
def posicion_falsa(f, p0, p1, TOL, n_max, cifras):
    
    i = 2
    
    p0 = redondear(p0, cifras)
    p1 = redondear(p1, cifras)
    
    fp0 = redondear(f(p0), cifras)
    sgm0 = signum(fp0)
    
    fp1 = redondear(f(p1), cifras)
    sgm1 = signum(fp1)
    
    if sgm0 * sgm1 > 0:
        print("f(p0) y f(p1) tienen el mismo signo: el intervalo no encierra una raíz")
        return -1
    
    filas = [(0, p0, fp0, sgm0, None)]
    filas.append((1, p1, fp1, sgm1, None))
    
    while i <= n_max:
        
        denominador = (fp1 - fp0)
                
        if denominador == 0:
            imprimir_tabla2(filas, cifras)
            print("La resta entre fp1 y fp0 genera indeterminación")
            return -1
        
        p = redondear(p1 - fp1 * (p1 - p0)/(denominador), cifras)
        error = redondear(abs(p - p1), cifras)
        
        fp  = redondear(f(p), cifras)
        sgmfp = signum(fp)
        
        filas.append((i, p, fp, sgmfp, error))
        
        
        if error <= TOL:
            imprimir_tabla2(filas, cifras)
            print("Procedimiento Exitoso")
            return p
        
        i = i + 1
        
        if sgmfp * sgm1 < 0:
            p0  = p1
            fp0 = fp1
        
        p1  = p
        fp1 = fp
        sgm1 = sgmfp
    
    imprimir_tabla2(filas, cifras)
    
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1
        

In [10]:
def posicion_falsat(f, p0, p1, TOL, n_max, cifras):
    
    i = 2
    
    p0 = truncar(p0, cifras)
    p1 = truncar(p1, cifras)
    
    fp0 = truncar(f(p0), cifras)
    sgm0 = signum(fp0)
    
    fp1 = truncar(f(p1), cifras)
    sgm1 = signum(fp1)
    
    if sgm0 * sgm1 > 0:
        print("f(p0) y f(p1) tienen el mismo signo: el intervalo no encierra una raíz")
        return -1
    
    filas = [(0, p0, fp0, sgm0, None)]
    filas.append((1, p1, fp1, sgm1, None))
    
    while i <= n_max:
        
        denominador = (fp1 - fp0)
                
        if denominador == 0:
            imprimir_tabla2(filas, cifras)
            print("La resta entre fp1 y fp0 genera indeterminación")
            return -1
        
        p = truncar(p1 - fp1 * (p1 - p0)/(denominador), cifras)
        error = truncar(abs(p - p1), cifras)
        
        fp  = truncar(f(p), cifras)
        sgmfp = signum(fp)
        
        filas.append((i, p, fp, sgmfp, error))
        
        
        if error <= TOL:
            imprimir_tabla2(filas, cifras)
            print("Procedimiento Exitoso")
            return p
        
        i = i + 1
        
        if sgmfp * sgm1 < 0:
            p0  = p1
            fp0 = fp1
        
        p1  = p
        fp1 = fp
        sgm1 = sgmfp
    
    imprimir_tabla2(filas, cifras)
    
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1
        

In [ ]:
def biseccionar1(f, a, b, TOL, n_max, decimales):
    i = 1
    
    a = redondear(a, decimales)
    b = redondear(b, decimales)
    p = redondear(a + (b-a) / 2, decimales)
    
    FA = redondear(f(a), decimales)
    FB = redondear(f(b), decimales)
    FP = redondear(f(p), decimales)
    
    cifras = [(0, a, b, p, FA, FB, FP, None)]
    
    while i <= n_max:
        p = redondear((a+b) / 2, decimales)
        FP = redondear(f(p), decimales)
        error = redondear((b-a)/2, decimales)
        
        # 1. Determinamos los nuevos valores de a y b sin sobrescribir todavía los actuales
        if signum(FA) * signum(FP) > 0:
            a_sig = p
            FA_sig = FP
            b_sig = b
            FB_sig = FB
        else:
            b_sig = p
            FB_sig = FP
            a_sig = a
            FA_sig = FA
        
        # 2. Guardamos la iteración actual en la tabla con sus valores correctos
        cifras.append((i, a, b, p, FA, FB, FP, error))
        
        # 3. Verificamos si ya se cumplió el criterio de parada
        if FP == 0 or error < TOL:
            imprimir_tabla3(cifras, decimales)
            print(f"Procedimiento completado exitosamente, el valor es: {p}")
            return p
        
        # 4. Actualizamos los valores para la siguiente iteración
        a, FA = a_sig, FA_sig
        b, FB = b_sig, FB_sig
        
        i = i + 1
    
    imprimir_tabla3(cifras, decimales)
    print(f"El proceso fracasó después de {n_max} iteraciones")
    return

In [12]:
posicion_falsa(lambda x: 230*x**4 + 18*x**3 + 9*x**2 -221 * x -9, -1, 0, 1e-6, 20, 4 )

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0         -1.0000           433.0000           1.0000              -         
1          0.0000           -9.0000           -1.0000              -         
2         -0.0204           -4.4880           -1.0000            0.0204      
3         -0.0304           -2.2736           -1.0000            0.0100      
4         -0.0355           -1.1436           -1.0000          5.1000e-03    
5         -0.0380           -0.5895           -1.0000          2.5000e-03    
6         -0.0393           -0.3013           -1.0000          1.3000e-03    
7         -0.0400           -0.1462           -1.0000          7.0000e-04    
8         -0.0403           -0.0797           -1.0000          3.0000e-04    
9         -0.0405           -0.0353           -1.0000          2.0000e-04    
10        -0.0406           -0.0131           -1.0000          1

-0.0406

In [13]:
secante(lambda x: 230*x**4 + 18*x**3 + 9*x**2 -221*x -9, -1, 0, 1e-6, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             -1.0000                 433.0000                     -            
1             0.0000                   -9.0000                     -            
2             -0.0204                  -4.4880                  0.0204          
3             -0.0407                9.0260e-03                 0.0203          
4             -0.0407                9.0260e-03                 0.0000          
Procedimiento Exitoso


-0.0407

In [14]:
secante(lambda x: math.tan(math.pi*x - 6), 0, 0.48, 1e-16, 10, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             0.0000                   0.2910                      -            
1             0.4800                   -4.4645                     -            
2             0.0294                   0.3943                   0.4506          
3             0.0660                   0.5341                   0.0366          
4             -0.0738                  0.0514                   0.1398          
5             -0.0887                4.5261e-03                 0.0149          
6             -0.0901                1.2781e-04               1.4000e-03        
7             -0.0901                1.2781e-04                 0.0000          
Procedimiento Exitoso


-0.0901

In [15]:
posicion_falsa(lambda x: math.tan(math.pi*x - 6), 0, 0.48, 1e-16, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          0.0000            0.2910            1.0000              -         
1          0.4800           -4.4645           -1.0000              -         
2          0.0294            0.3943            1.0000            0.4506      
3          0.0660            0.5341            1.0000            0.0366      
4          0.1102            0.7282            1.0000            0.0442      
5          0.1621            1.0142            1.0000            0.0519      
6          0.2209            1.4819            1.0000            0.0588      
7          0.2855            2.4280            1.0000            0.0646      
8          0.3540            5.6398            1.0000            0.0685      
9          0.4243           -22.0275          -1.0000            0.0703      
10         0.3683            7.6156            1.0000           

-1

In [16]:
biseccionar1(lambda x: math.tan(math.pi*x)-6, 0, 0.48, 1e-16, 20, 4)

n            a                 b                 p             sgm(f(a))         sgm(f(b))         sgm(f(p))           Error       
-----------------------------------------------------------------------------------------------------------------------------------
0          0.0000            0.4800            0.2400           -6.0000            9.8945           -5.0609              -         
1          0.0000            0.4800            0.2400           -6.0000            9.8945           -5.0609            0.2400      
2          0.2400            0.4800            0.3600           -5.0609            9.8945           -3.8749            0.1200      
3          0.3600            0.4800            0.4200           -3.8749            9.8945           -2.1053            0.0600      
4          0.4200            0.4800            0.4500           -2.1053            9.8945            0.3138            0.0300      
5          0.4200            0.4500            0.4350           -2.1053     

In [17]:
posicion_falsa(lambda x: math.tan(math.pi*x)-6, 0, 0.48, 1e-16, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          0.0000           -6.0000           -1.0000              -         
1          0.4800            9.8945            1.0000              -         
2          0.1812           -5.3601           -1.0000            0.2988      
3          0.2862           -4.7421           -1.0000            0.1050      
4          0.3490           -4.0525           -1.0000            0.0628      
5          0.3871           -3.2998           -1.0000            0.0381      
6          0.4103           -2.5458           -1.0000            0.0232      
7          0.4246           -1.8576           -1.0000            0.0143      
8          0.4334           -1.2905           -1.0000          8.8000e-03    
9          0.4388           -0.8631           -1.0000          5.4000e-03    
10         0.4421           -0.5632           -1.0000          3

0.4474

In [18]:
secante(lambda x: math.tan(math.pi*x)-6, 0, 0.48, 1e-16, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             0.0000                   -6.0000                     -            
1             0.4800                   9.8945                      -            
2             0.1812                   -5.3601                  0.2988          
3             0.2862                   -4.7421                  0.1050          
4             1.0919                   -5.7030                  0.8057          
5             -3.6900                  -4.5285                  4.7819          
6            -22.1275                  -6.4234                  18.4375         
7             40.3726                  -3.6364                  62.5001         
8            121.9210                  -6.2534                  81.5484         
9            -72.9413                  -5.8135                 194.8623         
10          -2648.1447      

-1

Ejericicio 7 a)

In [19]:
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), -1, 0, 1e-16, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             -1.0000                  1.3635                      -            
1             0.0000                   -1.0000                     -            
2             -0.4231                  -0.0373                  0.4231          
3             -0.4395                  0.0181                   0.0164          
4             -0.4341                -1.4568e-04              5.4000e-03        
5             -0.4341                -1.4568e-04                0.0000          
Procedimiento Exitoso


-0.4341

b)

In [20]:
metodo_newton(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), lambda x: (2*x)/(x**2 + 1) - 0.4*math.exp(0.4*x)*math.cos(math.pi*x) + math.exp(0.4*x)*math.pi*math.sin(math.pi*x), 0.5, 1e-16, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             0.5000                   0.2231                   4.6371                      -            
1             0.4519                 5.4652e-03                 4.3995                   0.0481          
2             0.4507                 1.8998e-04                 4.3925                 1.2000e-03        
3             0.4507                 1.8998e-04                 4.3925                   0.0000          

Procedimiento exitoso!


0.4507

In [21]:
posicion_falsa(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 1, 2, 1e-16, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          1.0000            2.1850            1.0000              -         
1          2.0000           -0.6161           -1.0000              -         
2          1.7801           -0.1432           -1.0000            0.2199      
3          1.7321            0.0543            1.0000            0.0480      
4          1.7453         -2.3773e-03         -1.0000            0.0132      
5          1.7447          1.6110e-04          1.0000          6.0000e-04    
6          1.7447          1.6110e-04          1.0000            0.0000      
Procedimiento Exitoso


1.7447

In [22]:
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 3, 4, 1e-16, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             3.0000                   5.6227                      -            
1             4.0000                   -2.1198                     -            
2             3.7262                   -0.1954                  0.2738          
3             3.6984                   0.1238                   0.0278          
4             3.7092                 -1.8329e-03                0.0108          
5             3.7090                 4.7564e-04               2.0000e-04        
6             3.7090                 4.7564e-04                 0.0000          
Procedimiento Exitoso


3.709

In [23]:
metodo_newton(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), lambda x: (2*x)/(x**2 + 1) - 0.4*math.exp(0.4*x)*math.cos(math.pi*x) + math.exp(0.4*x)*math.pi*math.sin(math.pi*x), 5/2, 1e-16, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             2.5000                   1.9810                   9.2294                      -            
1             2.2854                   0.2709                   6.2342                   0.2146          
2             2.2419                   0.0190                   5.3391                   0.0435          
3             2.2383                 -1.0417e-04                5.2623                 3.6000e-03        
4             2.2383                 -1.0417e-04                5.2623                   0.0000          

Procedimiento exitoso!


2.2383

In [24]:
metodo_newton(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), lambda x: (2*x)/(x**2 + 1) - 0.4*math.exp(0.4*x)*math.cos(math.pi*x) + math.exp(0.4*x)*math.pi*math.sin(math.pi*x), 24.5, 1e-16, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             24.5000                  6.3990                 56654.7621                    -            
1             24.4999                  0.7338                 56650.2270               1.0000e-04        
2             24.4999                  0.7338                 56650.2270                 0.0000          

Procedimiento exitoso!


24.4999

Busqueda de los 4 primeros ceros de la funcion

In [25]:
#[0, 1]
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 0, 1, 1e-6, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             0.0000                   -1.0000                     -            
1             1.0000                   2.1850                      -            
2             0.3140                   -0.5314                  0.6860          
3             0.4482                   -0.0108                  0.1342          
4             0.4510                 1.5080e-03               2.8000e-03        
5             0.4507                 1.8998e-04               3.0000e-04        
6             0.4507                 1.8998e-04                 0.0000          
Procedimiento Exitoso


0.4507

In [26]:
#[1, 2]
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 1, 2, 1e-6, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             1.0000                   2.1850                      -            
1             2.0000                   -0.6161                     -            
2             1.7801                   -0.1432                  0.2199          
3             1.7135                   0.1367                   0.0666          
4             1.7460                 -5.3344e-03                0.0325          
5             1.7448                 -2.6222e-04              1.2000e-03        
6             1.7447                 1.6110e-04               1.0000e-04        
7             1.7447                 1.6110e-04                 0.0000          
Procedimiento Exitoso


1.7447

In [27]:
#[2, 3]
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 2, 3, 1e-6, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             2.0000                   -0.6161                     -            
1             3.0000                   5.6227                      -            
2             2.0988                   -0.5173                  0.9012          
3             2.1747                   -0.2905                  0.0759          
4             2.2719                   0.1886                   0.0972          
5             2.2336                   -0.0246                  0.0383          
6             2.2380                 -1.6819e-03              4.4000e-03        
7             2.2383                 -1.0417e-04              3.0000e-04        
8             2.2383                 -1.0417e-04                0.0000          
Procedimiento Exitoso


2.2383

In [28]:
#[3, 4]
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 3, 4, 1e-6, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             3.0000                   5.6227                      -            
1             4.0000                   -2.1198                     -            
2             3.7262                   -0.1954                  0.2738          
3             3.6984                   0.1238                   0.0278          
4             3.7092                 -1.8329e-03                0.0108          
5             3.7090                 4.7564e-04               2.0000e-04        
6             3.7090                 4.7564e-04                 0.0000          
Procedimiento Exitoso


3.709

Vigésimo quinto término

In [29]:
#[0, 1]
secante(lambda x: math.log(x**2 + 1) - math.exp(0.4 * x) * math.cos(math.pi * x), 24, 25, 1e-6, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             24.0000                -14758.4237                   -            
1             25.0000                22032.9051                    -            
2             24.4011                -5293.1862                 0.5989          
3             24.5171                 981.3756                  0.1160          
4             24.4990                 -50.2330                  0.0181          
5             24.4999                  0.7338                 9.0000e-04        
6             24.4999                  0.7338                   0.0000          
Procedimiento Exitoso


24.4999

Metodo de Newton Ejercicios 1, 2

In [30]:
metodo_newton(lambda x: x**2 - 6, lambda x: 2*x, 1, 1e-3, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.0000                   -5.0000                  2.0000                      -            
1             3.5000                   6.2500                   7.0000                   2.5000          
2             2.6071                   0.7970                   5.2142                   0.8929          
3             2.4542                   0.0231                   4.9084                   0.1529          
4             2.4495                 5.0250e-05                 4.8990                 4.7000e-03        
5             2.4495                 5.0250e-05                 4.8990                   0.0000          

Procedimiento exitoso!


2.4495

In [31]:
metodo_newton(lambda x: -x**3 - math.cos(x), lambda x: -3*x**2 + math.sin(x), -1, 1e-3, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             -1.0000                  0.4597                   -3.8415                     -            
1             -0.8803                  0.0452                   -3.0957                  0.1197          
2             -0.8657                6.7998e-04                 -3.0099                  0.0146          
3             -0.8655                7.8124e-05                 -3.0087                2.0000e-04        

Procedimiento exitoso!


-0.8655

Ejercicio 3, literales a,b,c,d.

In [32]:
metodo_newton(lambda x: x**3 - 2*x**2 - 5, lambda x: 3*x**2 - 4*x, 2.5, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             2.5000                   -1.8750                  8.7500                      -            
1             2.7143                   0.2626                   11.2451                  0.2143          
2             2.6909                 2.7674e-03                 10.9592                  0.0234          
3             2.6906                 -5.1983e-04                10.9556                3.0000e-04        
4             2.6906                 -5.1983e-04                10.9556                  0.0000          

Procedimiento exitoso!


2.6906

In [33]:
metodo_newton(lambda x: x**3 + 3*x**2 - 1, lambda x: 3*x**2 + 6*x, -2.5, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             -2.5000                  2.1250                   3.7500                      -            
1             -3.0667                  -1.6273                  9.8137                   0.5667          
2             -2.9009                  -0.1661                  7.8403                   0.1658          
3             -2.8797                -2.3915e-03                7.5998                   0.0212          
4             -2.8794                -1.1211e-04                7.5964                 3.0000e-04        
5             -2.8794                -1.1211e-04                7.5964                   0.0000          

Procedimiento exitoso!


-2.8794

In [34]:
metodo_newton(lambda x: x - math.cos(x), lambda x: 1 + math.sin(x), math.pi/4, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             0.7854                   0.0783                   1.7071                      -            
1             0.7395                 6.9439e-04                 1.6739                   0.0459          
2             0.7391                 2.4881e-05                 1.6736                 4.0000e-04        
3             0.7391                 2.4881e-05                 1.6736                   0.0000          

Procedimiento exitoso!


0.7391

In [35]:
metodo_newton(lambda x: x - 0.8 - 0.2*math.sin(x), lambda x: 1 - 0.2*math.cos(x), math.pi/4, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             0.7854                   -0.1560                  0.8586                      -            
1             0.9671                 2.4514e-03                 0.8865                   0.1817          
2             0.9643                 -3.0025e-05                0.8860                 2.8000e-03        
3             0.9643                 -3.0025e-05                0.8860                   0.0000          

Procedimiento exitoso!


0.9643

Ejercicio 4a

In [36]:
metodo_newton(lambda x : 3*x - math.exp(x), lambda x : 3 - math.exp(x), 1.5, 1e-5, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.5000                   0.0183                   -1.4817                     -            
1             1.5124                 -4.0800e-04                -1.5376                  0.0124          
2             1.5121                 5.3083e-05                 -1.5362                3.0000e-04        
3             1.5121                 5.3083e-05                 -1.5362                  0.0000          

Procedimiento exitoso!


1.5121

In [37]:
secante(lambda x : 3*x - math.exp(x), 1, 2, 1e-5, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             1.0000                   0.2817                      -            
1             2.0000                   -1.3891                     -            
2             1.1686                   0.2883                   0.8314          
3             1.3115                   0.2228                   0.1429          
4             1.7976                   -0.6423                  0.4861          
5             1.4367                   0.1033                   0.3609          
6             1.4867                   0.0376                   0.0500          
7             1.5153                 -4.8862e-03                0.0286          
8             1.5120                 2.0668e-04               3.3000e-03        
9             1.5121                 5.3083e-05               1.0000e-04        
10            1.5121        

1.5121

In [38]:
posicion_falsa(lambda x : 3*x - math.exp(x), 1, 2, 1e-5, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          1.0000            0.2817            1.0000              -         
1          2.0000           -1.3891           -1.0000              -         
2          1.1686            0.2883            1.0000            0.8314      
3          1.3115            0.2228            1.0000            0.1429      
4          1.4067            0.1376            1.0000            0.0952      
5          1.4602            0.0738            1.0000            0.0535      
6          1.4874            0.0366            1.0000            0.0272      
7          1.5006            0.0174            1.0000            0.0132      
8          1.5068          8.1316e-03          1.0000          6.2000e-03    
9          1.5097          3.7270e-03          1.0000          2.9000e-03    
10         1.5110          1.7402e-03          1.0000          1

1.5121

Ejercicio 4 b

In [39]:
metodo_newton(lambda x : 2*x + 3* math.cos(x) - math.exp(x), lambda x : 2-3*math.sin(x) - math.exp(x), 1.5, 1e-5, 20, 4 )

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.5000                   -1.2695                  -5.4742                     -            
1             1.2681                   -0.1236                  -4.4177                  0.2319          
2             1.2401                 -1.6539e-03                -4.2934                  0.0280          
3             1.2397                 6.3079e-05                 -4.2916                4.0000e-04        
4             1.2397                 6.3079e-05                 -4.2916                  0.0000          

Procedimiento exitoso!


1.2397

In [40]:
secante(lambda x : 2*x + 3* math.cos(x) - math.exp(x), 1, 2, 1e-5, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             1.0000                   0.9026                      -            
1             2.0000                   -4.6375                     -            
2             1.1629                   0.3166                   0.8371          
3             1.2164                   0.0989                   0.0535          
4             1.2407                 -4.2308e-03                0.0243          
5             1.2397                 6.3079e-05               1.0000e-03        
6             1.2397                 6.3079e-05                 0.0000          
Procedimiento Exitoso


1.2397

In [41]:
posicion_falsa(lambda x : 2*x + 3* math.cos(x) - math.exp(x), 1, 2, 1e-5, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          1.0000            0.9026            1.0000              -         
1          2.0000           -4.6375           -1.0000              -         
2          1.1629            0.3166            1.0000            0.8371      
3          1.2164            0.0989            1.0000            0.0535      
4          1.2328            0.0296            1.0000            0.0164      
5          1.2377          8.6375e-03          1.0000          4.9000e-03    
6          1.2391          2.6373e-03          1.0000          1.4000e-03    
7          1.2395          9.2132e-04          1.0000          4.0000e-04    
8          1.2397          6.3079e-05          1.0000          2.0000e-04    
9          1.2397          6.3079e-05          1.0000            0.0000      
Procedimiento Exitoso


1.2397

Ejercicio 5

In [42]:
posicion_falsa(lambda x: 230*x**4 + 18*x**3 + 9*x**2 -221 * x -9, -1, 0, 1e-6, 20, 4 )

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0         -1.0000           433.0000           1.0000              -         
1          0.0000           -9.0000           -1.0000              -         
2         -0.0204           -4.4880           -1.0000            0.0204      
3         -0.0304           -2.2736           -1.0000            0.0100      
4         -0.0355           -1.1436           -1.0000          5.1000e-03    
5         -0.0380           -0.5895           -1.0000          2.5000e-03    
6         -0.0393           -0.3013           -1.0000          1.3000e-03    
7         -0.0400           -0.1462           -1.0000          7.0000e-04    
8         -0.0403           -0.0797           -1.0000          3.0000e-04    
9         -0.0405           -0.0353           -1.0000          2.0000e-04    
10        -0.0406           -0.0131           -1.0000          1

-0.0406

In [43]:
secante(lambda x: 230*x**4 + 18*x**3 + 9*x**2 -221 * x -9, -1, 0, 1e-6, 20, 4 )

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             -1.0000                 433.0000                     -            
1             0.0000                   -9.0000                     -            
2             -0.0204                  -4.4880                  0.0204          
3             -0.0407                9.0260e-03                 0.0203          
4             -0.0407                9.0260e-03                 0.0000          
Procedimiento Exitoso


-0.0407

In [44]:
f  = lambda x: math.copysign(abs(x)**(1/3), x)
df = lambda x: (1/3) * abs(x)**(-2/3)

In [45]:
metodo_newton(f, df, 1, 1e-5, 6, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.0000                   1.0000                   0.3333                      -            
1             -2.0003                  -1.2600                  0.2100                   3.0003          
2             3.9997                   1.5874                   0.1323                   6.0000          
3             -7.9988                  -1.9999                  0.0833                   11.9985         
4             16.0096                  2.5203                   0.0525                   24.0084         
5            -31.9961                  -3.1747                  0.0331                   48.0057         
6             63.9163                  3.9983                   0.0209                   95.9124         

El procedimiento falló después de 6 iteracion

-1

In [46]:
secante(f, 5, 0.5, 1e-16, 6, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             5.0000                   1.7100                      -            
1             0.5000                   0.7937                      -            
2             -3.3979                  -1.5034                  3.8979          
3             -0.8468                  -0.9461                  2.5511          
4             3.4841                   1.5160                   4.3309          
5             0.8174                   0.9350                   2.6667          
6             -3.4741                  -1.5145                  4.2915          

El procedimiento falló después de 6 iteraciones


-1

In [47]:
posicion_falsat(lambda x: math.sin(x) + math.log(x) - 1, 3, 4, 1e-50, 35, 11)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0      3.00000000000     0.23973229672     1.00000000000           -         
1      4.00000000000     -0.37050813418    -1.00000000000          -         
2      3.39284892409     -0.02695102074    -1.00000000000    0.60715107591   
3      3.35314760493   -8.09494342326e-05  -1.00000000000    0.03970131915   
4      3.35302839925   4.91023475213e-08   1.00000000000   1.19205680000e-04 
5      3.35302847151   2.20112816862e-12   1.00000000000   7.22600002056e-08 
6      3.35302847151   2.20112816862e-12   1.00000000000     0.00000000000   
Procedimiento Exitoso


3.35302847151

In [50]:
def secante1(f, p0, p1, TOL, n_max, cifras):
    i = 2
    
    p0 = truncar(p0, cifras)
    p1 = truncar(p1, cifras)
    fp0 = truncar(f(p0), cifras)
    fp1 = truncar(f(p1), cifras)
    
    filas = [(0, p0, fp0, None)]
    filas.append((1, p1, fp1, None))
    
    while i <= n_max:
        
        denominador = (fp1 - fp0)
        
        if denominador == 0:
            imprimir_tabla1(filas, cifras)
            print("La resta entre fp1 y fp0 genera indeterminación")
            return -1
        
        p = truncar(p1 - fp1*(p1 - p0)/denominador, cifras)
        error = truncar(abs(p - p1), cifras)
        fp = truncar(f(p), cifras) #lo calculo aqui para poder mostrarlo
        
        filas.append((i, p, fp, error))
        
        if error < TOL:
            imprimir_tabla1(filas, cifras)
            print("Procedimiento Exitoso")
            return p
        
        
        p0  = p1
        fp0 = fp1
        p1  = p
        fp1 = fp
        
        i = i +1
    
    imprimir_tabla1(filas, cifras)
    
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1

In [52]:
secante1(lambda x : x**4 -x -1, 1, 1.4, 1e-6, 20, 8)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0           1.00000000               -1.00000000                   -            
1           1.40000000               1.44159999                    -            
2           1.16382699               -0.32917499              0.23617300        
3           1.20772994               -0.08018211              0.04390294        
4           1.22186781             7.06457363e-03             0.01413787        
5           1.22072303             -1.32149270e-04          1.14477999e-03      
6           1.22074405             -2.17209615e-07          2.10199999e-05      
7           1.22074408             -2.89089243e-08          3.00000000e-08      
Procedimiento Exitoso


1.22074408